In [ ]:
import os
import sys
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import joblib

sys.path.append('..')
from utils import EALSTM_helper
from utils.EALSTM_helper import EALSTM, EASequenceDataset, make_sequences_with_static, evaluate

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Configuration

Entity-Aware LSTM for streamflow prediction in the Uinta Basin.
Static catchment attributes (area, elevation, slope, land cover) modulate
the LSTM input gate, allowing the model to condition predictions on basin characteristics.

Following Kratzert et al. (2019):
- Dynamic inputs: meteorological forcing + SWE (time-varying)
- Static inputs: catchment attributes (constant per basin)
- Spatial holdout: one basin withheld entirely for testing

Split strategy:
- Train: 1980–2016 (7 training sites)
- Validation: 2017–2024 (7 training sites)  
- Spatial test: held-out site, all years

In [ ]:
# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Paths
FILE_PATH  = "../data/merged/merged_all_sites.csv"
MODEL_DIR  = "model/EALSTM/"
DATE_COL   = "Date"
TARGET_COL = "flow_cms"

# Spatial holdout
TEST_SITE = "10154200"  # Provo River

# Temporal split
TRAIN_END_YEAR = 2016
VAL_START_YEAR = 2017

# Feature columns
DYNAMIC_COLS = [
    'SWE_cm_mean', 'prcp_mm_day', 'tmean_C',
    'tmax_C', 'tmin_C', 'srad_W_m2', 'vp_Pa', 'dayl_s'
]

STATIC_COLS = [
    'Area_km2', 'Average_Elevation_m',
    'Average_Slope', 'Evergreen_Forest'
]

# Hyperparameters
LOOKBACK_DAYS = 100
BATCH_SIZE    = 64
EPOCHS        = 50
PATIENCE      = 8
LEARNING_RATE = 1e-3
HIDDEN_SIZE   = 64
DROPOUT       = 0.2

## Load and Preview Data

In [ ]:
df = pd.read_csv(FILE_PATH)

# Clean column names
clean_cols = []
for c in df.columns:
    c = str(c).strip().replace('"', '')
    c = ''.join(ch if ch.isalnum() else '_' for ch in c)
    while '__' in c:
        c = c.replace('__', '_')
    c = c.strip('_')
    clean_cols.append(c)
df.columns = clean_cols

# Parse date and sort
df[DATE_COL] = pd.to_datetime(df[DATE_COL])
df['gage_id'] = df['gage_id'].astype(str)
df = df.sort_values(['gage_id', DATE_COL]).reset_index(drop=True)

print('Total rows:', len(df))
print('Sites:', sorted(df['gage_id'].unique()))
print('Date range:', df[DATE_COL].min().date(), 'to', df[DATE_COL].max().date())
df.head()

In [ ]:
# Verify all required columns are present
missing_dynamic = [c for c in DYNAMIC_COLS if c not in df.columns]
missing_static  = [c for c in STATIC_COLS  if c not in df.columns]

print("Missing dynamic cols:", missing_dynamic)
print("Missing static cols: ", missing_static)
print("\nStatic attribute values per site:")
print(df.groupby('gage_id')[STATIC_COLS].first().round(2))

In [ ]:
# Fill missing values per site
all_feature_cols = DYNAMIC_COLS + [TARGET_COL]
df[all_feature_cols] = (
    df.groupby('gage_id')[all_feature_cols]
    .transform(lambda x: x.interpolate(method='linear', limit_direction='both').ffill().bfill())
)

# Fill any remaining static attribute NaNs with 0
df[STATIC_COLS] = df[STATIC_COLS].fillna(0)

print("Missing values remaining:")
print(df[DYNAMIC_COLS + STATIC_COLS + [TARGET_COL]].isnull().sum())

## Spatial and Temporal Splits

The held-out site is withheld entirely from training.
The remaining sites are split temporally into train and validation sets.

In [ ]:
# Spatial split
spatial_test_df = df[df['gage_id'] == TEST_SITE].copy()
train_sites_df  = df[df['gage_id'] != TEST_SITE].copy()

# Temporal split on training sites
train_sites_df['year'] = train_sites_df[DATE_COL].dt.year
train_df = train_sites_df[train_sites_df['year'] <= TRAIN_END_YEAR].copy()
val_df   = train_sites_df[train_sites_df['year'] >= VAL_START_YEAR].copy()

print(f"Spatial test site ({TEST_SITE}): {len(spatial_test_df)} rows")
print(f"Training sites: {sorted(train_sites_df['gage_id'].unique())}")
print(f"Train rows:     {len(train_df)}")
print(f"Val rows:       {len(val_df)}")
print(f"Test rows:      {len(spatial_test_df)}")

## Scale Features

Three separate scalers:
- **feature_scaler**: dynamic met + SWE features
- **static_scaler**: static catchment attributes
- **target_scaler**: streamflow target

All scalers are fit on training data only to prevent data leakage.

In [ ]:
os.makedirs(MODEL_DIR, exist_ok=True)

feature_scaler = MinMaxScaler()
static_scaler  = MinMaxScaler()
target_scaler  = MinMaxScaler()

# Fit on training data only
feature_scaler.fit(train_df[DYNAMIC_COLS])
static_scaler.fit(train_df[STATIC_COLS])
target_scaler.fit(train_df[[TARGET_COL]])

# Save scalers
joblib.dump(feature_scaler, f"{MODEL_DIR}feature_scaler.pkl")
joblib.dump(static_scaler,  f"{MODEL_DIR}static_scaler.pkl")
joblib.dump(target_scaler,  f"{MODEL_DIR}target_scaler.pkl")

# Apply scaling to all splits
def scale_split(df):
    df = df.copy()
    df[DYNAMIC_COLS] = feature_scaler.transform(df[DYNAMIC_COLS])
    df[STATIC_COLS]  = static_scaler.transform(df[STATIC_COLS])
    df[TARGET_COL]   = target_scaler.transform(df[[TARGET_COL]])
    return df

train_scaled = scale_split(train_df)
val_scaled   = scale_split(val_df)
test_scaled  = scale_split(spatial_test_df)

print("Scaling complete.")

## Create Sequences

Using `make_sequences_with_static()` which processes each site separately,
avoiding sequences that bleed across basin boundaries.

In [ ]:
X_train, X_train_static, y_train, d_train = make_sequences_with_static(
    train_scaled, DATE_COL, DYNAMIC_COLS, STATIC_COLS, TARGET_COL, LOOKBACK_DAYS)

X_val, X_val_static, y_val, d_val = make_sequences_with_static(
    val_scaled, DATE_COL, DYNAMIC_COLS, STATIC_COLS, TARGET_COL, LOOKBACK_DAYS)

X_test, X_test_static, y_test, d_test = make_sequences_with_static(
    test_scaled, DATE_COL, DYNAMIC_COLS, STATIC_COLS, TARGET_COL, LOOKBACK_DAYS)

print('Train shape:  ', X_train.shape, X_train_static.shape, y_train.shape)
print('Val shape:    ', X_val.shape,   X_val_static.shape,   y_val.shape)
print('Test shape:   ', X_test.shape,  X_test_static.shape,  y_test.shape)

In [ ]:
train_loader = DataLoader(EASequenceDataset(X_train, X_train_static, y_train), batch_size=BATCH_SIZE, shuffle=False)
val_loader   = DataLoader(EASequenceDataset(X_val,   X_val_static,   y_val),   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(EASequenceDataset(X_test,  X_test_static,  y_test),  batch_size=BATCH_SIZE, shuffle=False)

## Define EA-LSTM Model

The key difference from a standard LSTM is the entity-aware input gate:
- Standard LSTM: all gates computed from dynamic inputs at every timestep
- EA-LSTM: input gate computed **once** from static attributes per basin

This allows the model to learn a different "personality" for each basin
based on its physical characteristics.

In [ ]:
model = EALSTM(
    dynamic_input_size = len(DYNAMIC_COLS),
    static_input_size  = len(STATIC_COLS),
    hidden_size        = HIDDEN_SIZE,
    dropout            = DROPOUT
).to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTrainable parameters: {total_params:,}")

## Train Model

In [ ]:
best_val_loss    = np.inf
best_state       = None
patience_counter = 0
history          = {'train_loss': [], 'val_loss': []}

for epoch in range(1, EPOCHS + 1):
    model.train()
    batch_losses = []

    for xb_dynamic, xb_static, yb in train_loader:
        xb_dynamic = xb_dynamic.to(device)
        xb_static  = xb_static.to(device)
        yb         = yb.to(device)

        optimizer.zero_grad()
        pred = model(xb_dynamic, xb_static)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()
        batch_losses.append(loss.item())

    train_loss = float(np.mean(batch_losses))
    val_loss, _, _ = evaluate(model, criterion, device, val_loader)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    print(f"Epoch {epoch:03d} | train loss = {train_loss:.5f} | val loss = {val_loss:.5f}")

    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        best_state       = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print("Early stopping triggered.")
            break

if best_state is not None:
    model.load_state_dict(best_state)
    torch.save(best_state, f"{MODEL_DIR}best_model_ealstm.pt")
    print("Best model saved.")

## Training History

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history['train_loss'], label='Train loss')
plt.plot(history['val_loss'],   label='Validation loss')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('EA-LSTM Training History')
plt.legend()
plt.tight_layout()
plt.show()

## Spatial Generalization Test

Evaluating on the held-out site — the model has never seen this basin during training.

In [ ]:
test_loss, test_pred_scaled, test_obs_scaled = evaluate(model, criterion, device, test_loader)

pred_flow = target_scaler.inverse_transform(test_pred_scaled.reshape(-1, 1)).ravel()
obs_flow  = target_scaler.inverse_transform(test_obs_scaled.reshape(-1, 1)).ravel()

rmse = math.sqrt(mean_squared_error(obs_flow, pred_flow))
mae  = mean_absolute_error(obs_flow, pred_flow)
r2   = r2_score(obs_flow, pred_flow)

print(f"Spatial Test Site: {TEST_SITE}")
print(f"Test MSE (scaled): {test_loss:.5f}")
print(f"RMSE:              {rmse:.4f} cms")
print(f"MAE:               {mae:.4f} cms")
print(f"R² / NSE:          {r2:.4f}")

## Observed vs Predicted — Time Series

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(d_test, obs_flow,  label='Observed',  linewidth=1)
plt.plot(d_test, pred_flow, label='Predicted', linewidth=1, alpha=0.8)
plt.xlabel('Date')
plt.ylabel('Streamflow (cms)')
plt.title(f'Observed vs Predicted Streamflow — Spatial Test Site {TEST_SITE}')
plt.legend()
plt.tight_layout()
plt.show()

## Observed vs Predicted — Scatter

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(obs_flow, pred_flow, alpha=0.4, s=5)
lims = [min(obs_flow.min(), pred_flow.min()), max(obs_flow.max(), pred_flow.max())]
plt.plot(lims, lims, 'k--', label='1:1 line')
plt.xlabel('Observed (cms)')
plt.ylabel('Predicted (cms)')
plt.title(f'Observed vs Predicted — Site {TEST_SITE}\nR²={r2:.3f}  RMSE={rmse:.3f} cms')
plt.legend()
plt.tight_layout()
plt.show()

## Performance Summary — All Sites

In [ ]:
site_results = []

for gage_id in sorted(train_sites_df['gage_id'].unique()):
    site_df = train_sites_df[train_sites_df['gage_id'] == gage_id].copy()
    site_scaled = scale_split(site_df)

    X_site, X_site_static, y_site, d_site = make_sequences_with_static(
        site_scaled, DATE_COL, DYNAMIC_COLS, STATIC_COLS, TARGET_COL, LOOKBACK_DAYS)

    site_loader = DataLoader(
        EASequenceDataset(X_site, X_site_static, y_site),
        batch_size=BATCH_SIZE, shuffle=False)

    site_loss, site_pred_scaled, site_obs_scaled = evaluate(model, criterion, device, site_loader)

    site_pred = target_scaler.inverse_transform(site_pred_scaled.reshape(-1, 1)).ravel()
    site_obs  = target_scaler.inverse_transform(site_obs_scaled.reshape(-1, 1)).ravel()

    site_results.append({
        "gage_id": gage_id,
        "split":   "train/val",
        "NSE":     r2_score(site_obs, site_pred),
        "RMSE":    math.sqrt(mean_squared_error(site_obs, site_pred)),
        "MAE":     mean_absolute_error(site_obs, site_pred),
    })

# Add spatial test site
site_results.append({
    "gage_id": TEST_SITE,
    "split":   "spatial test (held out)",
    "NSE":     r2,
    "RMSE":    rmse,
    "MAE":     mae,
})

results_df = pd.DataFrame(site_results).set_index("gage_id")
print(results_df.round(3))
results_df.round(3).to_csv(f"../data/ealstm_performance_summary.csv")

In [ ]:
from matplotlib.patches import Patch

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

metrics = ["NSE", "RMSE", "MAE"]
for ax, metric in zip(axes, metrics):
    colors = ["steelblue" if s == "train/val" else "tomato"
              for s in results_df["split"]]
    ax.bar(results_df.index, results_df[metric], color=colors)
    ax.set_title(metric)
    ax.set_xlabel("Gage ID")
    ax.tick_params(axis='x', rotation=45)

legend_elements = [
    Patch(facecolor="steelblue", label="Train/Val sites"),
    Patch(facecolor="tomato",    label="Spatial test (held out)")
]
axes[0].legend(handles=legend_elements)

plt.suptitle("EA-LSTM Performance by Site", fontsize=13)
plt.tight_layout()
plt.show()